# 🥘 Look and Cook: AI-Powered Recipe Assistant

This notebook allows you to run the **Look and Cook** ingredient detection and recipe generation pipeline directly in Jupyter. It uses **YOLOv8** for fast object detection and **LM Studio** for advanced vision analysis and culinary expertise.

In [ ]:
import io
import os
import base64
import json
import requests
from PIL import Image, ImageEnhance
from ultralytics import YOLO
from IPython.display import display, HTML
import ipywidgets as widgets

## ⚙️ Configuration
Set up your LM Studio hosts and model names here.

In [ ]:
LM_STUDIO_HOSTS = [
    "http://127.0.0.1:1234",
    "http://192.168.56.1:1234"
]
VISION_MODEL = "moondream-2b-2025-04-14"
RECIPE_MODEL = "meta-llama-3-8b-instruct"
YOLO_MODEL_PATH = "yolov8s-world.pt"

# Grounding classes for YOLO-World
GROUNDING_CLASSES = [
    "whole chicken", "raw chicken", "chicken breast", "meat", "tomato", "cherry tomato",
    "red onion", "onion", "garlic", "potato", "carrot", "rosemary", "herb",
    "broccoli", "egg", "milk", "cheese", "bread", "pepper", "lemon",
    "lettuce", "cucumber", "beef", "pork", "fish", "shrimp", "rice", "pasta",
    "vegetable", "fruit", "spice"
]

NON_FOOD_BLACKLIST = {
    "table", "tabletop", "counter", "countertop", "plate", "bowl", "cup", "fork", 
    "knife", "spoon", "glass", "background", "surface", "wooden", "wood", "board",
    "cutting board", "person", "hand", "finger", "cloth", "napkin", "kitchen",
    "indoor", "outdoor", "image", "photo", "picture", "assortment", "display"
}

## 🧠 Load YOLO Model

In [ ]:
print(f"Loading YOLO model: {YOLO_MODEL_PATH}...")
model = YOLO(YOLO_MODEL_PATH)

if "world" in YOLO_MODEL_PATH.lower():
    print(f"🌍 Setting YOLO-World classes: {len(GROUNDING_CLASSES)} items")
    model.set_classes(GROUNDING_CLASSES)

print("✅ YOLO model loaded!")

## 🔍 Detection Logic

In [ ]:
def detect_ingredients_yolo(image):
    """Detect food items using YOLOv8"""
    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(1.2)
    
    results = model(image, conf=0.15)
    detected_items = []

    for result in results:
        for box in result.boxes:
            class_id = int(box.cls[0])
            class_name = model.names[class_id]
            confidence = float(box.conf[0])
            
            min_conf = 0.35 if "world" in YOLO_MODEL_PATH.lower() else 0.45
            if confidence > min_conf:
                detected_items.append({
                    "name": class_name,
                    "confidence": round(confidence * 100, 1)
                })
    
    return list({item["name"]: item for item in detected_items}.values())

def detect_ingredients_vision(image_path):
    """Fallback: Use LM Studio Vision Model"""
    try:
        with open(image_path, "rb") as f:
            image_data = f.read()
        base64_image = base64.b64encode(image_data).decode('utf-8')
        
        prompt = "Identify the edible food ingredients in this image. List them ONLY as short names separated by commas. No descriptions."

        for host in LM_STUDIO_HOSTS:
            url = f"{host}/v1/chat/completions"
            payload = {
                "model": VISION_MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                        ]
                    }
                ],
                "temperature": 0.1,
            }
            try:
                response = requests.post(url, json=payload, timeout=60)
                if response.status_code == 200:
                    content = response.json()['choices'][0]['message']['content']
                    items = [i.strip().title() for i in content.split(',') if i.strip()]
                    return [{"name": name, "confidence": 100.0} for name in items]
            except: continue
        return []
    except Exception as e:
        print(f"Vision error: {e}")
        return []

def generate_recipes(ingredients_list):
    """Generate recipes using LM Studio"""
    ingredients_text = ", ".join([item["name"] for item in ingredients_list])
    prompt = f"Based on these ingredients: {ingredients_text}, suggest 3 distinct recipes. Format as: ### [Recipe Name]\n**Time**: [Time]\n**Description**: [Description]\n**Instructions**: [Steps]"

    for host in LM_STUDIO_HOSTS:
        url = f"{host}/v1/chat/completions"
        payload = {
            "model": RECIPE_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.7
        }
        try:
            response = requests.post(url, json=payload, timeout=120)
            if response.status_code == 200:
                return response.json()['choices'][0]['message']['content']
        except: continue
    return "Error: Could not connect to LM Studio."

## 🍳 Run Analysis
Upload an image or provide a path to analyze.

In [ ]:
def process_image(image_path):
    img = Image.open(image_path)
    display(img.resize((400, 300)))
    
    print("🔍 Detecting ingredients...")
    ingredients = detect_ingredients_yolo(img)
    
    if not ingredients:
        print("⚠️ YOLO found nothing, trying Vision model...")
        ingredients = detect_ingredients_vision(image_path)
        
    if ingredients:
        print(f"✅ Found: {', '.join([i['name'] for i in ingredients])}")
        print("\n🍳 Generating recipes...")
        recipes = generate_recipes(ingredients)
        
        # Simple HTML display for recipes
        display(HTML(f"<div style='background: #f8f9fa; padding: 20px; border-radius: 10px; border: 1px solid #dee2e6;'>{recipes.replace('\n', '<br>')}</div>"))
    else:
        print("❌ No ingredients detected.")

# Example usage:
# process_image('your_image.jpg')